<a href="https://colab.research.google.com/github/Girawale-Sanket/Gen-AI-LLM-Assignment-/blob/main/Chatbot_cum_AI_Powered_Search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Chatbot cum AI-Powered Search Engine
import os, sys, json, html, time, shutil, subprocess, threading
import requests

def install_ollama():
    if shutil.which("ollama") or os.path.exists("/usr/local/bin/ollama"):
        print(" Ollama binary already present.")
        return True

    print(" Installing Ollama from official tarball...")
    # zstd is needed to extract the .tar.zst bundle
    subprocess.run("apt-get update -qq && apt-get install -y -qq zstd curl",
                   shell=True, check=False)

    # Download the linux bundle and extract into /usr/local
    cmd = (
        "curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst "
        "| tar --zstd -x -C /usr/local"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(" Tarball install failed:")
        print(r.stderr[:500])
        return False

    os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")
    print(" Ollama installed.")
    return True

install_ollama()

def server_up():
    try:
        return requests.get("http://localhost:11434/api/tags", timeout=2).ok
    except Exception:
        return False

OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"

if not server_up():
    print(" Starting Ollama server in the background...")
    # nohup + & so it survives after this cell finishes
    subprocess.Popen(
        f"nohup {OLLAMA_BIN} serve > /tmp/ollama_serve.log 2>&1 &",
        shell=True,
        preexec_fn=os.setsid if hasattr(os, "setsid") else None
    )
    for i in range(40):                # wait up to 20s
        if server_up():
            break
        time.sleep(0.5)
        if i % 10 == 9:
            print(f"   still waiting... ({i//2}s)")

print(" Server:", "up" if server_up() else "NOT reachable")
if not server_up():
    print("   Check /tmp/ollama_serve.log for details.")

PREFERRED = "llama3.2"                     # change to a smaller model if slow

def list_models():
    try:
        return [m["name"] for m in requests.get(
            "http://localhost:11434/api/tags", timeout=3).json().get("models", [])]
    except Exception:
        return []

models = list_models()
if not models and server_up():
    print(f"  Pulling {PREFERRED} (first time only, ~2GB)...")
    subprocess.run([OLLAMA_BIN, "pull", PREFERRED], check=False)
    models = list_models()

MODEL = next((m for m in models if m == PREFERRED), None) \
     or next((m for m in models if m.startswith(PREFERRED.split(":")[0])), None) \
     or (models[0] if models else None)

if MODEL is None:
    print(" No model available. Check the logs above.")
else:
    print(f" Using model: {MODEL}")

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ipywidgets", "requests"], check=False)
    import ipywidgets as widgets
    from IPython.display import display

history = []

status_bar = widgets.HTML(value=(
    f"<div style='font:13px system-ui;color:#555;padding:4px 2px'>"
    f"{' Connected · using model <b>'+html.escape(MODEL)+'</b>' if MODEL else '⚠️ No model'}"
    f"</div>"))

chat = widgets.HTML(layout=widgets.Layout(
    width="100%", height="420px", overflow="auto",
    border="1px solid #e0e0e0", border_radius="14px",
    padding="14px", background="#fafafa"))

inp   = widgets.Text(placeholder="Ask me anything...",
                     layout=widgets.Layout(width="78%"))
send  = widgets.Button(description="Send",  button_style="success",
                       layout=widgets.Layout(width="11%"))
clear = widgets.Button(description="Clear", button_style="warning",
                       layout=widgets.Layout(width="11%"))

def bubble(role, text):
    user = role == "user"
    bg, fg, align = ("#0b93f6", "#fff", "flex-end") if user else ("#fff", "#111", "flex-start")
    return (f"<div style='display:flex;justify-content:{align};margin:6px 0'>"
            f"<div style='max-width:78%;padding:10px 14px;border-radius:16px;"
            f"background:{bg};color:{fg};font:14px/1.5 system-ui,sans-serif;"
            f"white-space:pre-wrap;box-shadow:0 1px 3px rgba(0,0,0,.08)'>"
            f"{html.escape(text)}</div></div>")

def render(streaming=""):
    body = "".join(bubble(m["role"], m["content"]) for m in history)
    if streaming:
        body += bubble("assistant", streaming)
    chat.value = body

def on_send(_=None):
    text = inp.value.strip()
    if not text:
        return
    if MODEL is None:
        chat.value += bubble("assistant",
            " No model available. Re-run the top of this cell.")
        return

    inp.value = ""
    history.append({"role": "user", "content": text})
    render("…")

    reply = ""
    try:
        with requests.post(
            "http://localhost:11434/api/chat",
            json={"model": MODEL, "messages": history, "stream": True},
            stream=True, timeout=300
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line:
                    continue
                chunk = json.loads(line)
                reply += chunk.get("message", {}).get("content", "")
                render(reply)
                if chunk.get("done"):
                    break
    except requests.exceptions.ConnectionError:
        history.pop(); render()
        chat.value += bubble("assistant",
            " Lost connection to Ollama.\nRun `ollama serve` in a terminal.")
        return
    except Exception as e:
        history.pop(); render()
        chat.value += bubble("assistant", f" Error: {e}")
        return

    history.append({"role": "assistant", "content": reply})
    render()

def on_clear(_=None):
    history.clear(); render()

send.on_click(on_send)
inp.on_submit(on_send)
clear.on_click(on_clear)

display(
    widgets.HTML("<h3 style='font-family:system-ui;margin:6px 0'>   ChatBot cum AI Powered Search Engine</h3>"),
    status_bar, chat,
    widgets.HBox([inp, send, clear])
)
render()

 Installing Ollama from official tarball...
 Ollama installed.
 Starting Ollama server in the background...
 Server: up
  Pulling llama3.2 (first time only, ~2GB)...
 Using model: llama3.2:latest


HTML(value="<h3 style='font-family:system-ui;margin:6px 0'>   ChatBot cum AI Powered Search Engine</h3>")

HTML(value="<div style='font:13px system-ui;color:#555;padding:4px 2px'> Connected · using model <b>llama3.2:l…

HTML(value='', layout=Layout(border='1px solid #e0e0e0', height='420px', overflow='auto', padding='14px', widt…

In [ ]:
def find_greatest_number():
    num1 = int(input("Enter first number: "))
    num2 = int(input("Enter second number: "))
    num3 = int(input("Enter third number: "))

    if num1 >= num2 and num1 >= num3:
        return num1
    elif num2 >= num1 and num2 >= num3:
        return num2
    else:
        return num3

print("The greatest number is: ", find_greatest_number())

Enter first number: 12
Enter second number: 10
Enter third number: 3
The greatest number is:  12
